Air Traffic Control

Chromosome - list of the rules like the hight, distance, altitude of each plane flying in the air avoiding crash.

---
Basic Imports

In [ ]:
import numpy as np
import pandas as pd

In [44]:
def closest_pass(p1, p2, v1, v2, time_window, minimum_distance):
    delta_p = p1 - p2
    delta_v = v1 - v2
    a = np.dot(delta_v, delta_v) # Speed difference between the planes
    b = 2 * np.dot(delta_p, delta_v)# Relative speed between the planes
    c = np.dot(delta_p, delta_p) # Distance between the planes

    #print ("a:", a, "b:", b, "c:", c)

    if a == 0:
        return np.sqrt(c)
    t = -b / (2 * a)

    #print("Time of closest approach:", t)
    if t < 0:
        return 0
    if t > time_window:
        return 0    
        
    closest_distance = np.sqrt(c + t * (b + a * t))
    
    #print("Closest distance:", closest_distance)
    if closest_distance < minimum_distance:
        return 100
    return 0


def convert_long_lat_alt_to_cartesian(long, lat, alt):
    y=lat * 111111
    x = long * 111111 * np.cos(np.radians(lat))
    z = alt
    return np.array([x, y, z])


def get_cartesian_velocity(vertrate, speed, heading):
    heading_rad = np.radians(heading)

    v_z = speed * np.cos(heading_rad)
    v_x = speed * np.sin(heading_rad)
    v_y = vertrate 
    
    return np.array([v_x, v_y, v_z])

## Test

In [45]:
#plane 1
lat1 =49.5354448739
long1 =2.83373033678
speed1 =181.779353382
head_1 =66.4820383263
vertrate1 = 4.55168
alt1 = 4838.7

#plane 2
lat2 =51.7063293457
long2 =-3.28788138725
speed2 = 236.560348391
head_2 =316.762391024
vertrate2 = 0.32512
alt2 = 11841.48

#plane 3
lat3 =49.542388916
long3 =2.85810771741
speed3 = 181.985246682
head_3 =246.3335256222
vertrate3 = 5.20192
alt3 = 4892.04

In [46]:
pos1 = convert_long_lat_alt_to_cartesian(long1, lat1, alt1)
vel1 = get_cartesian_velocity(vertrate1, speed1, head_1)

pos2 = convert_long_lat_alt_to_cartesian(long2, lat2, alt2)
vel2 = get_cartesian_velocity(vertrate2, speed2, head_2) 

pos3 = convert_long_lat_alt_to_cartesian(long3, lat3, alt3)
vel3 = get_cartesian_velocity(vertrate3, speed3, head_3)

In [47]:
print("Plane 1 position:", pos1, "velocity:", vel1)
print("Plane 3 position:", pos3, "velocity:", vel3)

closest_pass(pos1, pos3, vel1, vel3, 60, 6000)

Plane 1 position: [2.04336159e+05 5.50393282e+06 4.83870000e+03] velocity: [166.679856   4.55168   72.536604]
Plane 3 position: [2.06064693e+05 5.50470437e+06 4.89204000e+03] velocity: [-166.679856    5.20192   -73.051048]


100

# Import Dataset

In [48]:
flights_df = pd.read_csv("Data/new_flights.csv")
print(flights_df.head())

         time  icao24        lat       lon    velocity     heading  vertrate  \
0  1527548410  4ca1d2  49.535445  2.833730  181.779353   66.482038   4.55168   
1  1527548410  4ca5ea  51.706329 -3.287881  236.560348  316.762391   0.32512   
2  1527548410  4ca8d7  53.679060 -5.311567  162.050677  269.818109  -4.87680   
3  1527548410  4ca303  53.778488 -6.063354  123.908406  261.163409  -2.60096   
4  1527548410  4ca97c  50.252563 -3.978424  226.215014  338.101807   0.00000   

   callsign  onground  alert    spi  squawk  baroaltitude  geoaltitude  \
0  RYR52GH      False  False  False   674.0       4701.54      4838.70   
1  RYR17XT      False  False  False  5701.0      11582.40     11841.48   
2  RYR73UF      False  False  False  2522.0       3817.62      3992.88   
3  RYR39BN      False  False  False  2002.0       2141.22      2270.76   
4  RYR74UE      False  False  False  7474.0      11582.40     11833.86   

   lastposupdate   lastcontact  
0   1.527548e+09  1.527548e+09  
1   1.52

In [49]:
positions = []
velocities = []

for _, row in flights_df.iterrows():
    pos = convert_long_lat_alt_to_cartesian(row["lon"], row["lat"], row["geoaltitude"])
    vel = get_cartesian_velocity(row["vertrate"], row["velocity"], row["heading"])
    positions.append(pos)
    velocities.append(vel)

flights_df["converted_position"] = positions
flights_df["cartesian_velocity"] = velocities

print(flights_df.head())

         time  icao24        lat       lon    velocity     heading  vertrate  \
0  1527548410  4ca1d2  49.535445  2.833730  181.779353   66.482038   4.55168   
1  1527548410  4ca5ea  51.706329 -3.287881  236.560348  316.762391   0.32512   
2  1527548410  4ca8d7  53.679060 -5.311567  162.050677  269.818109  -4.87680   
3  1527548410  4ca303  53.778488 -6.063354  123.908406  261.163409  -2.60096   
4  1527548410  4ca97c  50.252563 -3.978424  226.215014  338.101807   0.00000   

   callsign  onground  alert    spi  squawk  baroaltitude  geoaltitude  \
0  RYR52GH      False  False  False   674.0       4701.54      4838.70   
1  RYR17XT      False  False  False  5701.0      11582.40     11841.48   
2  RYR73UF      False  False  False  2522.0       3817.62      3992.88   
3  RYR39BN      False  False  False  2002.0       2141.22      2270.76   
4  RYR74UE      False  False  False  7474.0      11582.40     11833.86   

   lastposupdate   lastcontact  \
0   1.527548e+09  1.527548e+09   
1   1.

In [50]:
TIME_WINDOW = 100      
MIN_DISTANCE = 6000   

conflict_pairs = []
n = len(flights_df)

for i in range(n):
    for j in range(i + 1, n):
        p1 = flights_df.iloc[i]["converted_position"]
        p2 = flights_df.iloc[j]["converted_position"]
        v1 = flights_df.iloc[i]["cartesian_velocity"]
        v2 = flights_df.iloc[j]["cartesian_velocity"]

        score = closest_pass(p1, p2, v1, v2, TIME_WINDOW, MIN_DISTANCE)

        if score > 0:
            conflict_pairs.append({
                "plane_A": flights_df.iloc[i]["callsign"].strip(),
                "plane_B": flights_df.iloc[j]["callsign"].strip(),
                "conflict_score": score
            })

conflicts_df = pd.DataFrame(conflict_pairs)

if conflicts_df.empty:
    print("No conflicts detected within the time window.")
else:
    print(f"Detected {len(conflicts_df)} potential conflict(s):")
    print(conflicts_df.to_string(index=False))


Detected 106 potential conflict(s):
plane_A plane_B  conflict_score
RYR74UE RYR74UE      100.000000
RYR66CU RYR66CU      100.000000
RYR66CU RYR66CU      100.000000
 RYR271  RYR271     3372.793277
 EIN529  EIN529      100.000000
EIN76HJ EIN76HJ      100.000000
 RY52GH  RYR52G     1685.010475
RYR66CU RYR66CU     2134.194295
RYR74UE RYR74UE      100.000000
RYR74UE RYR74UE      100.000000
RYR74UE RYR74UE    19747.101414
 EIN529  EIN529     2224.939695
 EIN529  EIN529     6900.073343
RYR17XT RYR17XT     4685.806474
RYR17XT RYR17XT     4590.109171
 YR52GH    52GH      100.000000
RYR87FC RYR87FC      100.000000
 EIN4JR  EIN4JR     2363.012035
RYR74UE RYR74UE     2206.923397
RYR74UE RYR74UE     4410.559370
RYR74UE RYR74UE    13168.150992
RYR74UE RYR74UE    15375.694264
 EIN529  EIN529     4675.142621
 EIN529  EIN529      100.000000
 RY72GH    52GH      100.000000
 RY72GH   RY2GH      100.000000
RYR66CU RYR66CU      100.000000
RYR74UE RYR74UE     2203.640453
RYR74UE RYR74UE    10961.227922
RYR7